In [53]:
import pandas as pd
import os
from pathlib import Path

def load_csv_safely(filepath, description=""):
    """安全載入CSV檔案，包含錯誤處理"""
    try:
        if not os.path.exists(filepath):
            print(f"警告: {filepath} 檔案不存在")
            return None
        
        df = pd.read_csv(filepath)
        print(f"✓ 成功載入 {description}: {df.shape[0]} 列, {df.shape[1]} 欄")
        return df
    except Exception as e:
        print(f"✗ 載入 {filepath} 時發生錯誤: {e}")
        return None

def standardize_column_names(df, mappings, df_name=""):
    """標準化欄位名稱"""
    original_columns = df.columns.tolist()
    
    for old_name, new_name in mappings.items():
        if old_name in df.columns and new_name not in df.columns:
            df = df.rename(columns={old_name: new_name})
            print(f"  {df_name}: 重新命名 '{old_name}' -> '{new_name}'")
    
    return df

def validate_merge_keys(df1, df2, keys, df1_name="", df2_name=""):
    """驗證合併鍵的有效性"""
    missing_keys_df1 = [key for key in keys if key not in df1.columns]
    missing_keys_df2 = [key for key in keys if key not in df2.columns]
    
    if missing_keys_df1:
        print(f"警告: {df1_name} 缺少合併鍵: {missing_keys_df1}")
    if missing_keys_df2:
        print(f"警告: {df2_name} 缺少合併鍵: {missing_keys_df2}")
    
    return len(missing_keys_df1) == 0 and len(missing_keys_df2) == 0

def merge_with_info(df1, df2, keys, how='left', df1_name="", df2_name="", suffixes=('', '_right')):
    """執行合併並顯示資訊"""
    if not validate_merge_keys(df1, df2, keys, df1_name, df2_name):
        print(f"✗ 無法合併 {df1_name} 和 {df2_name}: 缺少必要的合併鍵")
        return df1
    
    original_rows = len(df1)
    merged_df = pd.merge(df1, df2, how=how, on=keys, suffixes=suffixes)
    new_rows = len(merged_df)
    
    print(f"✓ 合併 {df1_name} + {df2_name}: {original_rows} -> {new_rows} 列")
    
    # 檢查是否有新的欄位被加入
    new_columns = set(merged_df.columns) - set(df1.columns)
    if new_columns:
        print(f"  新增欄位: {list(new_columns)}")
    
    return merged_df

def main():
    print("=== F1 數據整合工具 ===\n")
    
    # 1. 載入所有CSV檔案
    print("1. 載入數據檔案...")
    file_configs = {
        'drivers': ('./data/drivers_updated.csv', '車手數據'),
        'laps': ('./data/fastest_laps_updated.csv', '最快圈速數據'),
        'teams': ('./data/teams_updated.csv', '車隊數據'),
        'winners': ('./data/winners.csv', '比賽勝者數據')
    }
    
    data = {}
    for key, (filepath, description) in file_configs.items():
        df = load_csv_safely(filepath, description)
        if df is not None:
            data[key] = df
        else:
            print(f"跳過 {key} 數據...")
    
    if not data:
        print("✗ 沒有成功載入任何數據檔案")
        return
    
    print(f"\n成功載入 {len(data)} 個數據檔案\n")
    
    # 2. 標準化欄位名稱
    print("2. 標準化欄位名稱...")
    column_mappings = {
        'Car': 'Team',
        'Winner': 'Driver'
    }
    
    for key, df in data.items():
        data[key] = standardize_column_names(df, column_mappings, key)
        print(f"  {key} 欄位: {list(data[key].columns)}")
    
    print()
    
    # 3. 開始合併數據
    print("3. 合併數據...")
    
    # 從drivers開始，如果沒有drivers就用其他數據作為起點
    if 'drivers' in data:
        result_df = data['drivers'].copy()
        current_name = 'drivers'
    else:
        # 找第一個可用的數據集
        first_key = list(data.keys())[0]
        result_df = data[first_key].copy()
        current_name = first_key
        print(f"使用 {first_key} 作為起始數據集")
    
    # 合併 laps 數據
    if 'laps' in data and 'drivers' in data:
        merge_keys = ['Driver', 'Team', 'year']
        result_df = merge_with_info(
            result_df, data['laps'], merge_keys, 
            df1_name=current_name, df2_name='laps'
        )
        current_name += '+laps'
    
    # 合併 teams 數據
    if 'teams' in data:
        merge_keys = ['Team', 'year']
        result_df = merge_with_info(
            result_df, data['teams'], merge_keys,
            df1_name=current_name, df2_name='teams',
            suffixes=('', '_team')
        )
        current_name += '+teams'
    
    # 合併 winners 數據（如果有Grand Prix欄位）
    if 'winners' in data:
        if 'Grand Prix' in data['winners'].columns:
            merge_keys = ['Grand Prix', 'Driver', 'Team', 'year']
            result_df = merge_with_info(
                result_df, data['winners'], merge_keys,
                df1_name=current_name, df2_name='winners',
                suffixes=('', '_winner')
            )
            current_name += '+winners'
        else:
            print("  winners 數據缺少 'Grand Prix' 欄位，跳過合併")
    
    # 4. 數據品質檢查
    print(f"\n4. 數據品質檢查...")
    print(f"最終數據集形狀: {result_df.shape}")
    print(f"總欄位數: {len(result_df.columns)}")
    
    # 檢查缺失值
    missing_summary = result_df.isnull().sum()
    missing_cols = missing_summary[missing_summary > 0]
    if len(missing_cols) > 0:
        print("缺失值統計:")
        for col, count in missing_cols.items():
            percentage = (count / len(result_df)) * 100
            print(f"  {col}: {count} ({percentage:.1f}%)")
    else:
        print("✓ 沒有發現缺失值")
    
    # 檢查重複列
    duplicates = result_df.duplicated().sum()
    if duplicates > 0:
        print(f"發現 {duplicates} 個重複列")
    else:
        print("✓ 沒有發現重複列")
    
    # 5. 輸出結果
    print(f"\n5. 輸出結果...")
    output_path = './data/f1_merged_all.csv'
    
    try:
        result_df.to_csv(output_path, index=False, encoding='utf-8')
        print(f"✓ 成功輸出到 {output_path}")
        print(f"檔案大小: {os.path.getsize(output_path) / 1024:.1f} KB")
    except Exception as e:
        print(f"✗ 輸出檔案時發生錯誤: {e}")
    
    # 顯示前幾列作為預覽
    print(f"\n數據預覽 (前5列):")
    print(result_df.head())
    
    return result_df

if __name__ == "__main__":
    merged_data = main()

=== F1 數據整合工具 ===

1. 載入數據檔案...
✓ 成功載入 車手數據: 1661 列, 7 欄
✓ 成功載入 最快圈速數據: 1108 列, 6 欄
✓ 成功載入 車隊數據: 695 列, 4 欄
✓ 成功載入 比賽勝者數據: 1110 列, 7 欄

成功載入 4 個數據檔案

2. 標準化欄位名稱...
  drivers: 重新命名 'Car' -> 'Team'
  drivers 欄位: ['Pos', 'Driver', 'Nationality', 'Team', 'PTS', 'year', 'Code']
  laps: 重新命名 'Car' -> 'Team'
  laps 欄位: ['Grand Prix', 'Driver', 'Team', 'Time', 'year', 'Code']
  teams 欄位: ['Pos', 'Team', 'PTS', 'year']
  winners: 重新命名 'Car' -> 'Team'
  winners: 重新命名 'Winner' -> 'Driver'
  winners 欄位: ['Grand Prix', 'Date', 'Driver', 'Team', 'Laps', 'Time', 'Name Code']

3. 合併數據...
✓ 合併 drivers + laps: 1661 -> 2284 列
  新增欄位: ['Grand Prix', 'Time', 'Code_right']
✓ 合併 drivers+laps + teams: 2284 -> 2284 列
  新增欄位: ['PTS_team', 'Pos_team']
警告: winners 缺少合併鍵: ['year']
✗ 無法合併 drivers+laps+teams 和 winners: 缺少必要的合併鍵

4. 數據品質檢查...
最終數據集形狀: (2284, 12)
總欄位數: 12
缺失值統計:
  Team: 11 (0.5%)
  Grand Prix: 1224 (53.6%)
  Time: 1225 (53.6%)
  Code_right: 1224 (53.6%)
  Pos_team: 264 (11.6%)
  PTS_team: 264 (11.6%)


In [54]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
import gradio as gr
import warnings
warnings.filterwarnings('ignore')

class F1PredictionSystem:
    def __init__(self):
        self.driver_skill_model = None
        self.team_skill_model = None
        self.ranking_model = None
        self.scalers = {}
        self.encoders = {}
        self.driver_names = []
        self.team_names = []
        self.circuit_names = []
        self.active_drivers = {}  # 活躍車手及其當前車隊
        self.driver_team_history = {}  # 車手-車隊歷史記錄
        self.training_data = None
        
    def prepare_data(self, df):
        print("準備數據中...")
        self.training_data = df.copy()
        
        df_clean = df.copy()
        
        # Define a placeholder for missing values
        nan_placeholder = "Unknown_Value" # Anda bisa memilih placeholder lain

        # Clean data and fill NaNs consistently
        # Driver
        df_clean['Driver'] = df_clean['Driver'].str.strip().fillna(nan_placeholder)
        
        # Team - This is the crucial one for your current error
        df_clean['Team'] = df_clean['Team'].str.strip().fillna(nan_placeholder)
        
        # Grand Prix
        if 'Grand Prix' in df_clean.columns:
            df_clean['Grand Prix'] = df_clean['Grand Prix'].str.strip().fillna(nan_placeholder)
        # Else: 'Grand Prix' column doesn't exist, will be handled later

        # Identify active drivers (now uses data with NaNs filled)
        self._identify_active_drivers(df_clean) 
        
        # Establish encoders
        # .unique() will now include nan_placeholder if any NaNs were present
        # No longer need .dropna() here because we used .fillna()
        self.driver_names = sorted(df_clean['Driver'].unique().tolist())
        self.team_names = sorted(df_clean['Team'].unique().tolist())
        
        if 'Grand Prix' in df_clean.columns:
            self.circuit_names = sorted(df_clean['Grand Prix'].unique().tolist())
        else:
            # Original default list if 'Grand Prix' column is missing
            self.circuit_names = ['Monaco', 'Silverstone', 'Monza']
            # If you expect 'Unknown_Value' to be a possible circuit input later
            # and the column might be missing, you might consider adding nan_placeholder here:
            # if nan_placeholder not in self.circuit_names:
            # self.circuit_names.append(nan_placeholder)
            # self.circuit_names.sort()

        self.encoders['driver'] = LabelEncoder().fit(self.driver_names)
        self.encoders['team'] = LabelEncoder().fit(self.team_names)
        self.encoders['circuit'] = LabelEncoder().fit(self.circuit_names)
        
        # Process data for encoding
        # df_clean already has NaNs filled with nan_placeholder
        df_processed = df_clean.copy() 
        
        df_processed['driver_encoded'] = self.encoders['driver'].transform(df_processed['Driver'])
        df_processed['team_encoded'] = self.encoders['team'].transform(df_processed['Team']) # This should now work
        
        if 'Grand Prix' in df_processed.columns:
            df_processed['circuit_encoded'] = self.encoders['circuit'].transform(df_processed['Grand Prix'])
        else:
            # If 'Grand Prix' column doesn't exist, assign random encoded circuits
            # Ensure the random choice range is from the fitted classes
            if len(self.encoders['circuit'].classes_) > 0:
                df_processed['circuit_encoded'] = np.random.choice(
                    len(self.encoders['circuit'].classes_), # Choose an index
                    size=len(df_processed)
                )
            else:
                # This case should ideally not happen if circuit_names is always populated
                df_processed['circuit_encoded'] = 0 
                print("Warning: Circuit encoder has no classes. Defaulting circuit_encoded to 0.")

        df_processed = self.create_skill_features(df_processed)
        return df_processed

    def _identify_active_drivers(self, df):
        """識別活躍車手並確定他們的當前車隊"""
        print("識別活躍車手...")
        
        # 如果沒有年份欄位，使用數據順序
        if 'year' not in df.columns:
            df = df.copy()
            df['year'] = range(len(df))
        
        # 找出每個車手最近的記錄
        latest_records = df.loc[df.groupby('Driver')['year'].idxmax()]
        
        # 建立車手歷史記錄
        for driver in df['Driver'].unique():
            driver_data = df[df['Driver'] == driver]
            team_history = driver_data.groupby('Team').size().sort_values(ascending=False)
            self.driver_team_history[driver] = team_history.to_dict()
        
        # 確定活躍車手標準：最近幾年有記錄
        current_max_year = df['year'].max()
        activity_threshold = current_max_year - 5  # 最近5年內有記錄視為活躍
        
        active_drivers = {}
        for _, record in latest_records.iterrows():
            driver = record['Driver']
            team = record['Team']
            last_year = record['year']
            
            # 如果車手在活躍期內有記錄，視為活躍車手
            if last_year >= activity_threshold:
                active_drivers[driver] = {
                    'current_team': team,
                    'last_active_year': last_year,
                    'total_races': len(df[df['Driver'] == driver])
                }
        
        # 至少保留20個活躍車手
        if len(active_drivers) < 20:
            # 按最後活躍年份排序，選取前20名
            all_drivers = []
            for _, record in latest_records.iterrows():
                all_drivers.append({
                    'driver': record['Driver'],
                    'team': record['Team'],
                    'year': record['year'],
                    'races': len(df[df['Driver'] == record['Driver']])
                })
            
            all_drivers.sort(key=lambda x: (x['year'], x['races']), reverse=True)
            active_drivers = {}
            for driver_info in all_drivers[:max(20, len(active_drivers))]:
                active_drivers[driver_info['driver']] = {
                    'current_team': driver_info['team'],
                    'last_active_year': driver_info['year'],
                    'total_races': driver_info['races']
                }
        
        self.active_drivers = active_drivers
        print(f"識別出 {len(active_drivers)} 位活躍車手")
        
        # 顯示活躍車手示例
        print("活躍車手示例:")
        for i, (driver, info) in enumerate(list(active_drivers.items())[:5]):
            print(f"  {driver} ({info['current_team']}) - {info['total_races']} 場比賽")

    def create_skill_features(self, df):
        """創建車手和車隊實力特徵"""
        print("計算實力特徵...")
        
        df = df.sort_values(['year', 'Driver']).reset_index(drop=True)
        df['driver_skill'] = 50.0
        df['team_skill'] = 50.0

        # 根據成績計算分數
        if 'Pos' in df.columns:
            # 位置越前分數越高，DNF或退賽給較低分數
            df['position_score'] = df['Pos'].apply(lambda x: 
                101 - float(x) if pd.notna(x) and str(x).replace('.','').isdigit() 
                else 30)  # DNF等情況給30分
        elif 'Position' in df.columns:
            df['position_score'] = df['Position'].apply(lambda x: 
                101 - float(x) if pd.notna(x) and str(x).replace('.','').isdigit() 
                else 30)
        else:
            # 如果沒有位置信息，根據其他因素估算
            df['position_score'] = 50 + np.random.normal(0, 15, len(df))

        # 計算車手實力（使用指數移動平均）
        for driver in df['Driver'].unique():
            driver_mask = df['Driver'] == driver
            driver_data = df[driver_mask].sort_values('year')
            
            if len(driver_data) > 0:
                # 使用指數移動平均，近期表現權重更高
                ema_scores = driver_data['position_score'].ewm(span=5, adjust=False).mean()
                df.loc[driver_mask, 'driver_skill'] = ema_scores.values

        # 計算車隊實力
        for team in df['Team'].unique():
            team_mask = df['Team'] == team
            team_data = df[team_mask].sort_values('year')
            
            if len(team_data) > 0:
                # 按年計算車隊平均表現
                yearly_avg = team_data.groupby('year')['position_score'].mean()
                team_ema = yearly_avg.ewm(span=3, adjust=False).mean()
                
                for year in team_ema.index:
                    year_mask = (df['Team'] == team) & (df['year'] == year)
                    df.loc[year_mask, 'team_skill'] = team_ema[year]

        # 標準化實力值到0-100範圍
        scaler = MinMaxScaler(feature_range=(0, 100))
        df['driver_skill'] = scaler.fit_transform(df[['driver_skill']]).flatten()
        df['team_skill'] = scaler.fit_transform(df[['team_skill']]).flatten()
        
        return df

    def build_skill_prediction_model(self, input_dim):
        """建立實力預測模型"""
        model = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(input_dim,)),
            tf.keras.layers.Dense(128, activation='relu'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(64, activation='relu'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dropout(0.2),
            tf.keras.layers.Dense(32, activation='relu'),
            tf.keras.layers.Dense(2, activation='sigmoid')  # 輸出車手和車隊實力
        ])
        
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
            loss='huber',  # 對異常值更穩健
            metrics=['mae']
        )
        return model

    def build_ranking_model(self, input_dim):
        """建立排名預測模型"""
        model = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(input_dim,)),
            tf.keras.layers.Dense(256, activation='relu'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dropout(0.4),
            tf.keras.layers.Dense(128, activation='relu'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(64, activation='relu'),
            tf.keras.layers.Dropout(0.2),
            tf.keras.layers.Dense(32, activation='relu'),
            tf.keras.layers.Dense(1, activation='linear')  # 預測排名位置
        ])
        
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
            loss='huber',
            metrics=['mae']
        )
        return model

    def train_models(self, df):
        """訓練預測模型"""
        print("開始訓練模型...")
        df_processed = self.prepare_data(df)
        
        # 準備實力預測數據
        skill_features = ['driver_encoded', 'team_encoded', 'year']
        X_skill = df_processed[skill_features].values
        y_skill = df_processed[['driver_skill', 'team_skill']].values / 100.0  # 標準化到0-1
        
        self.scalers['skill'] = StandardScaler()
        X_skill_scaled = self.scalers['skill'].fit_transform(X_skill)
        
        # 訓練實力預測模型
        print("訓練實力預測模型...")
        X_skill_train, X_skill_test, y_skill_train, y_skill_test = train_test_split(
            X_skill_scaled, y_skill, test_size=0.2, random_state=42)
        
        self.driver_skill_model = self.build_skill_prediction_model(X_skill_scaled.shape[1])
        
        early_stopping = tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=15, restore_best_weights=True)
        
        reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=8, min_lr=1e-6)
        
        self.driver_skill_model.fit(
            X_skill_train, y_skill_train,
            validation_data=(X_skill_test, y_skill_test),
            epochs=100,
            batch_size=64,
            callbacks=[early_stopping, reduce_lr],
            verbose=1
        )
        
        # 準備排名預測數據
        ranking_features = ['driver_encoded', 'team_encoded', 'circuit_encoded', 'year', 'driver_skill', 'team_skill']
        X_ranking = df_processed[ranking_features].values
        
        # 目標變量：實際排名
        if 'Pos' in df_processed.columns:
            y_ranking = pd.to_numeric(df_processed['Pos'], errors='coerce').fillna(12)
        elif 'Position' in df_processed.columns:
            y_ranking = pd.to_numeric(df_processed['Position'], errors='coerce').fillna(12)
        else:
            # 如果沒有位置信息，根據綜合實力計算排名
            df_processed['combined_skill'] = (df_processed['driver_skill'] + df_processed['team_skill']) / 2
            y_ranking = df_processed.groupby(['year', 'circuit_encoded'])['combined_skill'].rank(
                method='min', ascending=False).values
        
        self.scalers['ranking'] = StandardScaler()
        X_ranking_scaled = self.scalers['ranking'].fit_transform(X_ranking)
        
        # 訓練排名預測模型
        print("訓練排名預測模型...")
        X_rank_train, X_rank_test, y_rank_train, y_rank_test = train_test_split(
            X_ranking_scaled, y_ranking, test_size=0.2, random_state=42)
        
        self.ranking_model = self.build_ranking_model(X_ranking_scaled.shape[1])
        
        self.ranking_model.fit(
            X_rank_train, y_rank_train,
            validation_data=(X_rank_test, y_rank_test),
            epochs=100,
            batch_size=64,
            callbacks=[early_stopping, reduce_lr],
            verbose=1
        )
        
        print("模型訓練完成！")

    def predict_driver_performance(self, driver_name, circuit_name, year):
        """預測單個車手的表現"""
        if driver_name not in self.active_drivers:
            return None
            
        current_team = self.active_drivers[driver_name]['current_team']
        
        try:
            # 編碼輸入特徵
            driver_encoded = self.encoders['driver'].transform([driver_name])[0]
            team_encoded = self.encoders['team'].transform([current_team])[0]
            circuit_encoded = self.encoders['circuit'].transform([circuit_name])[0]
            
            # 預測實力
            skill_features = np.array([[driver_encoded, team_encoded, year]])
            skill_features_scaled = self.scalers['skill'].transform(skill_features)
            skills = self.driver_skill_model.predict(skill_features_scaled, verbose=0)[0]
            
            driver_skill = float(skills[0] * 100)
            team_skill = float(skills[1] * 100)
            
            # 預測排名
            ranking_features = np.array([[driver_encoded, team_encoded, circuit_encoded, 
                                       year, driver_skill, team_skill]])
            ranking_features_scaled = self.scalers['ranking'].transform(ranking_features)
            predicted_position = self.ranking_model.predict(ranking_features_scaled, verbose=0)[0][0]
            
            return {
                'driver': driver_name,
                'team': current_team,
                'driver_skill': driver_skill,
                'team_skill': team_skill,
                'predicted_position': float(predicted_position),
                'confidence': self._calculate_confidence(driver_name, current_team)
            }
            
        except Exception as e:
            print(f"預測 {driver_name} 時發生錯誤: {e}")
            return None

    def _calculate_confidence(self, driver_name, team_name):
        """計算預測信心度"""
        if driver_name not in self.driver_team_history:
            return 0.5
            
        # 基於歷史數據量和車隊熟悉度計算信心度
        total_races = self.active_drivers.get(driver_name, {}).get('total_races', 0)
        team_races = self.driver_team_history[driver_name].get(team_name, 0)
        
        data_confidence = min(total_races / 50, 1.0)  # 基於總比賽數
        team_confidence = min(team_races / 20, 1.0)   # 基於與當前車隊的經驗
        
        return (data_confidence + team_confidence) / 2

    def predict_race_ranking(self, circuit_name, year, selected_drivers, top_n=10):
        """預測比賽排名"""
        print(f"預測 {circuit_name} {year} 年比賽，選定 {len(selected_drivers)} 位車手")
        
        results = []
        for driver_name in selected_drivers:
            result = self.predict_driver_performance(driver_name, circuit_name, year)
            if result:
                results.append(result)
        
        if not results:
            return []
        
        # 按預測排名排序
        results.sort(key=lambda x: x['predicted_position'])
        
        # 重新分配最終排名並限制輸出數量
        final_results = []
        for i, result in enumerate(results[:top_n]):
            result['final_position'] = i + 1
            final_results.append(result)
            
        return final_results

def create_gradio_interface(prediction_system):
    """創建Gradio界面"""
    
    def format_prediction_results(circuit, year, selected_drivers, top_n):
        if not selected_drivers:
            return "請選擇至少一位車手進行預測"
            
        try:
            results = prediction_system.predict_race_ranking(
                circuit, int(year), selected_drivers, int(top_n))
            
            if not results:
                return "預測失敗，請檢查選擇的車手"
            
            # 格式化輸出
            output = f"## 🏁 {circuit} {year}年 比賽預測結果 (前{len(results)}名)\n\n"
            
            output += "| 排名 | 車手 | 當前車隊 | 車手實力 | 車隊實力 | 綜合實力 | 信心度 |\n"
            output += "|------|------|----------|----------|----------|----------|--------|\n"
            
            for result in results:
                combined_skill = (result['driver_skill'] + result['team_skill']) / 2
                confidence_pct = result['confidence'] * 100
                
                output += f"| {result['final_position']} | {result['driver']} | {result['team']} | "
                output += f"{result['driver_skill']:.1f} | {result['team_skill']:.1f} | "
                output += f"{combined_skill:.1f} | {confidence_pct:.0f}% |\n"
            
            # 添加預測邏輯說明
            output += "\n\n### 📊 預測邏輯說明\n"
            output += """
**實力計算:**
- **車手實力**: 基於歷史比賽成績，使用指數移動平均突出近期表現
- **車隊實力**: 基於車隊歷年平均表現，反映技術水平和資源
- **綜合實力**: 車手實力與車隊實力的平均值

**排名預測:**
- 結合車手實力、車隊實力、賽道特性和年份因素
- 使用深度學習模型分析歷史數據模式
- 考慮車手與當前車隊的配合程度

**信心度:**
- 基於車手的歷史比賽數據量
- 考慮車手與當前車隊的合作經驗
- 數值越高表示預測越可靠
            """
            
            return output
            
        except Exception as e:
            return f"預測過程中發生錯誤: {str(e)}"

    # 獲取活躍車手列表
    active_driver_list = list(prediction_system.active_drivers.keys())
    active_driver_list.sort()
    
    with gr.Blocks(title="F1 比賽預測系統", theme=gr.themes.Soft()) as interface:
        gr.Markdown("# 🏎️ F1 比賽預測系統")
        gr.Markdown("""
        ### 🎯 系統特色
        - **智能車隊匹配**: 自動為車手匹配當前車隊，無需手動選擇
        - **活躍車手篩選**: 只顯示近期活躍的車手，確保預測相關性  
        - **靈活排名輸出**: 可自定義顯示前幾名的預測結果
        - **透明預測邏輯**: 詳細說明實力計算和排名預測方法
        """)
        
        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown("### ⚙️ 比賽設定")
                
                circuit_dropdown = gr.Dropdown(
                    choices=prediction_system.circuit_names,
                    label="🏁 選擇賽道",
                    value=prediction_system.circuit_names[0] if prediction_system.circuit_names else None
                )
                
                year_slider = gr.Slider(
                    minimum=2020,
                    maximum=2030,
                    step=1,
                    value=2024,
                    label="📅 比賽年份"
                )
                
                drivers_dropdown = gr.Dropdown(
                    choices=active_driver_list,
                    label="🏎️ 選擇參賽車手",
                    multiselect=True,
                    value=active_driver_list[:10] if len(active_driver_list) >= 10 else active_driver_list,
                    info=f"從 {len(active_driver_list)} 位活躍車手中選擇"
                )
                select_all_btn = gr.Button("全選/取消全選")

                # 全選/取消全選 callback
                def toggle_select_all_drivers(selected, choices):
                    if selected and set(selected) == set(choices):
                        # 如果已經全選→改成全取消
                        return []
                    return choices  # 否則就全選

                select_all_btn.click(
                    fn=toggle_select_all_drivers,
                    inputs=[drivers_dropdown, gr.State(active_driver_list)],
                    outputs=drivers_dropdown
                )
                
                top_n_slider = gr.Slider(
                    minimum=3,
                    maximum=20,
                    step=1,
                    value=10,
                    label="🏆 顯示排名數量",
                    info="選擇要顯示的前N名結果"
                )
                
                predict_button = gr.Button("🚀 開始預測", variant="primary", size="lg")
                
                # 系統統計
                gr.Markdown(f"""
                **📈 系統數據:**
                - 活躍車手: {len(active_driver_list)} 位
                - 可選賽道: {len(prediction_system.circuit_names)} 個
                - 訓練數據: {len(prediction_system.training_data) if prediction_system.training_data is not None else 0} 筆記錄
                """)
                
            with gr.Column(scale=2):
                gr.Markdown("### 🏆 預測結果")
                results_output = gr.Markdown(
                    value="選擇車手和比賽參數，然後點擊「開始預測」查看結果",
                    elem_classes=["prediction-output"]
                )
        
        predict_button.click(
            fn=format_prediction_results,
            inputs=[circuit_dropdown, year_slider, drivers_dropdown, top_n_slider],
            outputs=[results_output]
        )
        
        # 活躍車手信息展示
        with gr.Row():
            gr.Markdown("### 👥 活躍車手一覽")
            
        active_drivers_info = "| 車手 | 當前車隊 | 總比賽數 | 最後活躍年 |\n|------|----------|----------|------------|\n"
        for driver, info in list(prediction_system.active_drivers.items())[:15]:
            active_drivers_info += f"| {driver} | {info['current_team']} | {info['total_races']} | {info['last_active_year']} |\n"
        
        if len(prediction_system.active_drivers) > 15:
            active_drivers_info += f"\n*還有 {len(prediction_system.active_drivers) - 15} 位活躍車手...*"
            
        gr.Markdown(active_drivers_info)
    
    return interface

In [ ]:
def main():
    print("載入F1數據...")
    try:
        df = pd.read_csv('./f1_merged_all.csv')
        print(f"成功載入數據: {df.shape}")
    except FileNotFoundError:
        print("未找到 f1_merged_all.csv，創建示例數據...")
        np.random.seed(42)
        drivers = ['Hamilton', 'Verstappen', 'Leclerc', 'Russell', 'Sainz', 'Norris', 'Piastri', 'Alonso']
        teams = ['Mercedes', 'Red Bull', 'Ferrari', 'McLaren', 'Aston Martin']
        circuits = ['Monaco', 'Silverstone', 'Monza', 'Spa', 'Suzuka']
        years = [2020, 2021, 2022, 2023, 2024]
        data = []
        for year in years:
            for circuit in circuits:
                for i, driver in enumerate(drivers):
                    team = teams[i % len(teams)]
                    position = np.random.randint(1, 21)
                    data.append({
                        'Driver': driver,
                        'Team': team,
                        'Grand Prix': circuit,
                        'year': year,
                        'Position': position,
                        'Fastest Lap Time': f"1:{np.random.randint(15, 45)}.{np.random.randint(100, 999)}"
                    })
        df = pd.DataFrame(data)
        df.to_csv('f1_demo_data.csv', index=False)
        print("創建了示例數據檔案: f1_demo_data.csv")
    prediction_system = F1PredictionSystem()
    prediction_system.train_models(df)
    interface = create_gradio_interface(prediction_system)
    print("啟動Gradio界面...")
    interface.launch()

if __name__ == '__main__':
    main()


載入F1數據...
成功載入數據: (2284, 12)
開始訓練模型...
準備數據中...
識別活躍車手...
識別出 34 位活躍車手
活躍車手示例:
  Alexander  Albon (Williams Mercedes) - 5 場比賽
  Antonio  Giovinazzi (Alfa Romeo Racing Ferrari) - 4 場比賽
  Carlos  Sainz (Ferrari) - 11 場比賽
  Charles  Leclerc (Ferrari) - 13 場比賽
  Daniel  Ricciardo (RB Honda RBPT) - 23 場比賽
計算實力特徵...
訓練實力預測模型...
Epoch 1/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0846 - mae: 0.3512 - val_loss: 0.0402 - val_mae: 0.2488 - learning_rate: 0.0010
Epoch 2/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0210 - mae: 0.1666 - val_loss: 0.0289 - val_mae: 0.2079 - learning_rate: 0.0010
Epoch 3/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0174 - mae: 0.1470 - val_loss: 0.0229 - val_mae: 0.1827 - learning_rate: 0.0010
Epoch 4/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0152 - mae: 0.1366 - val_loss: 0.0188 - val_mae: 0.1639 - learning_rate: 0.0010
Epoch 5/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0151 - mae: 0.1371 - val_loss: 0.0167 - val_mae: 0.

預測 Brazil 2024 年比賽，選定 10 位車手
預測 Brazil 2024 年比賽，選定 34 位車手
